In [ ]:
import QuantLib as ql
import plotly.express as px

In [ ]:
import numpy as np
import polars as pl

In [ ]:
layout_dict = dict(
    margin=dict(l=20, r=20, t=40, b=20),
    width=600,
    height=400,
    paper_bgcolor="LightSteelBlue",
    title_font_size=14,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
)

# Zero coupon bond valuation

In [ ]:
from enum import Enum, auto


class CurveMode(Enum):
    FLAT = auto()
    FLATINCEPTION = auto()  # theta(t) & r(t) force forward/zero coupon curve evolution.
    ROLLUP = auto()
    ROLLDOWN = auto()

In [ ]:
def get_zc_bond() -> ql.FixedRateBond:
    settlement_days = 2
    face_amount = 100

    issue_date = ql.Date(30, 6, 2019)
    maturity_date = ql.Date(15, 9, 2023)
    calendar = ql.UnitedStates(ql.UnitedStates.Settlement)
    payment_convention = ql.Following

    # zcbond = ql.ZeroCouponBond(settlement_days, calendar, face_amount, maturity_date, payment_convention, face_amount, issue_date)
    # return zcbond

    czcbond = ql.CallableZeroCouponBond(
        settlement_days,
        face_amount,
        calendar,
        maturity_date,
        ql.Actual360(),
        payment_convention,
        face_amount,
        issue_date,
        ql.CallabilitySchedule(),
    )
    return czcbond

In [ ]:
def calc_from_curve(
    bond: ql.Bond,
    d: ql.Date,
    curve_mode: CurveMode = CurveMode.FLAT,
    flat_rate: float | None = None,
    grid_points: int | None = None,
) -> dict:
    ql.Settings.instance().evaluationDate = d
    settle_date = bond.calendar().advance(d, bond.settlementDays(), ql.Days)

    if flat_rate is None:
        flat_rate = 0.02
    if grid_points is None:
        grid_points = 100

    dates = ql.MakeSchedule(
        d,
        bond.maturityDate(),  # maturity date will be skipped
        ql.Period("1D"),
        calendar=bond.calendar(),
    )
    # dates = list(dates)
    dates = [bond.calendar().advance(x, bond.settlementDays(), ql.Days) for x in dates]

    a = 0.2
    sigma = 0.05
    if curve_mode == CurveMode.FLAT:
        curve = ql.FlatForward(
            bond.settlementDays(),
            ql.UnitedStates(ql.UnitedStates.Settlement),
            ql.QuoteHandle(ql.SimpleQuote(flat_rate)),
            ql.Actual360(),
            ql.Compounded,
            ql.Semiannual,
        )
    elif curve_mode == CurveMode.FLATINCEPTION:
        # zero rates consistent with flat term structure as of bond issue date
        # (!) for settle_date = bond issue date, this is flat structure but generally not so
        zero_rates = [flat_rate] + [
            flat_rate
            + ((sigma**2) / (4.0 * a**3))
            * (1.0 - np.exp(-2.0 * a * (settle_date - bond.issueDate()) / 360.0))
            * np.square(1.0 - np.exp(-a * (d_ - settle_date) / 360.0))
            / ((d_ - settle_date) / 360.0)
            for d_ in dates[1:]
        ]
        curve = ql.ZeroCurve(
            dates,
            zero_rates,
            ql.Actual360(),
            bond.calendar(),
            ql.Linear(),
            ql.Compounded,
            ql.Semiannual,
        )
    elif curve_mode == CurveMode.ROLLUP:  # this curve is "lifting up" over time
        # forwards = [flat_rate] + [flat_rate - (sigma**2)/(2*a**2)*np.square(1. - np.exp(-a*(d_-d)/360)) for d_ in dates[1:]]
        # curve = ql.ForwardCurve(list(dates), forwards, ql.Actual360(), bond.calendar()) # this doesn't admit semiannual compounding
        zero_rates = [flat_rate] + [
            flat_rate
            - ((sigma**2) / (2.0 * a**2))
            * (
                ((d_ - settle_date) / 360.0)
                - (2.0 / a) * (1.0 - np.exp(-a * ((d_ - settle_date) / 360.0)))
                + (0.5 / a) * (1.0 - np.exp(-2 * a * ((d_ - settle_date) / 360.0)))
            )
            / ((d_ - settle_date) / 360.0)
            for d_ in dates[1:]
        ]
        curve = ql.ZeroCurve(
            dates,
            zero_rates,
            ql.Actual360(),
            bond.calendar(),
            ql.Linear(),
            ql.Compounded,
            ql.Semiannual,
        )
    elif (
        curve_mode == CurveMode.ROLLDOWN
    ):  # this curve is "shifting down" over time (intuitive situation)
        zero_rates = [flat_rate] + [
            flat_rate
            + ((sigma**2) / (2 * a**2))
            * (
                ((d_ - settle_date) / 360.0)
                - (0.5 / a) * (1.0 - np.exp(-2 * a * ((d_ - settle_date) / 360.0)))
            )
            / ((d_ - settle_date) / 360.0)
            for d_ in dates[1:]
        ]
        curve = ql.ZeroCurve(
            dates,
            zero_rates,
            ql.Actual360(),
            bond.calendar(),
            ql.Linear(),
            ql.Compounded,
            ql.Semiannual,
        )

    else:
        NotImplementedError(f"Unrecognized curve_mode: {curve_mode}")

    # forwards =  [curve.forwardRate(d_, d_, ql.Actual360(), ql.Compounded, ql.Semiannual).rate() for d_ in dates] # transform back
    ts_handle = ql.YieldTermStructureHandle(curve)

    # bond_engine = ql.DiscountingBondEngine(ts_handle)
    # bond.setPricingEngine(bond_engine)
    engine = ql.TreeCallableFixedRateBondEngine(
        ql.HullWhite(ts_handle, a, sigma), grid_points
    )
    bond.setPricingEngine(engine)

    effective_duration = bond.effectiveDuration(
        0.0, ts_handle, ql.Actual360(), ql.Compounded, ql.Semiannual, 5e-4
    )
    effective_convexity = bond.effectiveConvexity(
        0.0, ts_handle, ql.Actual360(), ql.Compounded, ql.Semiannual, 5e-4
    )

    return {
        "date": d.to_date(),
        "flat_rate": flat_rate,
        "price": bond.cleanPrice(),
        "effective_duration": effective_duration,
        "effective_convexity": effective_convexity,
    }

## Theoretical valuation reconciliation

In [ ]:
zcb = get_zc_bond()
flat_rate = 0.01
valuation_date = ql.Date(15, 9, 2021)
print(
    100.0
    * pow(
        1 + flat_rate * 0.5,
        2
        * (
            zcb.calendar().advance(valuation_date, zcb.settlementDays(), ql.Days)
            - zcb.maturityDate()
        )
        / 360.0,
    )
)
calc_from_curve(zcb, valuation_date, curve_mode=CurveMode.FLAT, flat_rate=flat_rate)

In [ ]:
settle_date = zcb.calendar().advance(valuation_date, zcb.settlementDays(), ql.Days)
a = 0.2
sigma = 0.05
z = flat_rate + ((sigma**2) / (4 * a**3)) * (
    1.0 - np.exp(-2.0 * a * (settle_date - zcb.issueDate()) / 360.0)
) * np.square(1.0 - np.exp(-a * (zcb.maturityDate() - settle_date) / 360.0)) / (
    (zcb.maturityDate() - settle_date) / 360.0
)
# z = curve.forwardRate(zcb.calendar().advance(valuation_date, zcb.settlementDays(), ql.Days), zcb.maturityDate(), ql.Actual360(), ql.Compounded, ql.Semiannual).rate()
print(
    100.0
    * pow(
        1 + z * 0.5,
        2 * (settle_date - zcb.maturityDate()) / 360.0,
    )
)
calc_from_curve(
    zcb, valuation_date, curve_mode=CurveMode.FLATINCEPTION, flat_rate=flat_rate
)

In [ ]:
# theoretical zcb valuation under Hull White
# this is tautology, given that curve was defined via zcb, but it is a non-trivial consistency check
settle_date = zcb.calendar().advance(valuation_date, zcb.settlementDays(), ql.Days)
a = 0.2
sigma = 0.05
z = flat_rate - ((sigma**2) / (2 * a**2)) * (
    ((zcb.maturityDate() - settle_date) / 360.0)
    - (2.0 / a) * (1.0 - np.exp(-a * ((zcb.maturityDate() - settle_date) / 360.0)))
    + (0.5 / a) * (1.0 - np.exp(-2 * a * ((zcb.maturityDate() - settle_date) / 360.0)))
) / ((zcb.maturityDate() - settle_date) / 360.0)
# z = curve.forwardRate(zcb.calendar().advance(valuation_date, zcb.settlementDays(), ql.Days), zcb.maturityDate(), ql.Actual360(), ql.Compounded, ql.Semiannual).rate()
print(
    100.0
    * pow(
        1 + z * 0.5,
        2 * (settle_date - zcb.maturityDate()) / 360.0,
    )
)
calc_from_curve(zcb, valuation_date, curve_mode=CurveMode.ROLLUP, flat_rate=flat_rate)

In [ ]:
# theoretical zcb valuation under Hull White
# this is tautology, given that curve was defined via zcb, but it is a non-trivial consistency check
settle_date = zcb.calendar().advance(valuation_date, zcb.settlementDays(), ql.Days)
a = 0.2
sigma = 0.05
z = flat_rate + ((sigma**2) / (2 * a**2)) * (
    ((zcb.maturityDate() - settle_date) / 360.0)
    - (0.5 / a) * (1.0 - np.exp(-2 * a * ((zcb.maturityDate() - settle_date) / 360.0)))
) / ((zcb.maturityDate() - settle_date) / 360.0)
# z = curve.forwardRate(zcb.calendar().advance(valuation_date, zcb.settlementDays(), ql.Days), zcb.maturityDate(), ql.Actual360(), ql.Compounded, ql.Semiannual).rate()
print(
    100.0
    * pow(
        1 + z * 0.5,
        2 * (settle_date - zcb.maturityDate()) / 360.0,
    )
)
calc_from_curve(zcb, valuation_date, curve_mode=CurveMode.ROLLDOWN, flat_rate=flat_rate)

In [ ]:
# manual check on effective convexity with +-1bp
flat_rate = 0.01
grid_points = 100
px0 = calc_from_curve(
    zcb,
    valuation_date,
    curve_mode=CurveMode.ROLLUP,
    flat_rate=flat_rate,
    grid_points=grid_points,
)
px_p = calc_from_curve(
    zcb,
    valuation_date,
    curve_mode=CurveMode.ROLLUP,
    flat_rate=flat_rate + 1e-4,
    grid_points=grid_points,
)
px_m = calc_from_curve(
    zcb,
    valuation_date,
    curve_mode=CurveMode.ROLLUP,
    flat_rate=flat_rate - 1e-4,
    grid_points=grid_points,
)
print(f"Effective convexity: {px0['effective_convexity']}")
print(
    f'Empirical convexity: {(px_m["price"] - 2 * px0["price"] + px_p["price"]) / (px0["price"] * (1e-4) ** 2)}'
)
# Note (!) this shows that effective convexity shifts PDE: theta changes to r+-1bp, consequently convexity != true gamma

# theoretical PDE convexity
settle_date = zcb.calendar().advance(valuation_date, zcb.settlementDays(), ql.Days)
a = 0.2
print(
    f"PDE convexity: {np.square(1.0 - np.exp(-a * ((zcb.maturityDate() - settle_date) / 360.0))) / np.square(
    a
)}"
)

## Rollup example PnL reconciliation

In [ ]:
# yield check
bond = get_zc_bond()
sched = ql.MakeSchedule(
    ql.Date(30, 6, 2019),
    ql.Date(15, 9, 2023),  # maturity date will be skipped
    ql.Period("1D"),
    calendar=ql.UnitedStates(ql.UnitedStates.Settlement),
)

res = pl.DataFrame(
    schema=pl.Schema(
        {
            "date": pl.Date,
            "flat_rate": pl.Float64,
            "price": pl.Float64,
            "effective_duration": pl.Float64,
            "effective_convexity": pl.Float64,
        }
    )
)

for i, d in enumerate(sched):
    try:
        row = calc_from_curve(bond, d, curve_mode=CurveMode.ROLLUP, flat_rate=flat_rate)
        res.extend(pl.from_dicts(row))
    except RuntimeError:
        print(d)

res = res.with_columns(
    settle_date=pl.col("date")
    .cast(pl.Int64)
    .add(25569)
    .map_elements(
        lambda x: bond.calendar()
        .advance(ql.Date(x), bond.settlementDays(), ql.Days)
        .to_date(),
        return_dtype=pl.Date,
    )
)

# add coupons
res = res.join(
    pl.from_dicts(
        [
            {"date": cf.date().to_date(), "coupon": cf.amount()}
            for cf in bond.cashflows()
        ]
    ),
    on="date",
    how="left",
)
res = res.with_columns(pl.col("coupon").fill_null(0.0))

In [ ]:
a = 0.2
# this is known from closed form solution for ZCB under Hull White
res = res.with_columns(
    pde_convexity=(
        1.0
        - (
            -a
            * pl.col("settle_date").map_elements(
                lambda x: (zcb.maturityDate().to_date() - x).days
            )
        )
        .truediv(360.0)
        .exp()
    )
    .truediv(a)
    .pow(2)
)

In [ ]:
# Check PDE
# V_t + 0.5*s**2*V_xx(x, t) + (theta - a*x) * V_x = x*V
# note that for the first PDE, V_x term is multiplied by 0

freq = 2
res = (
    res.with_columns(
        ddays=(pl.col("settle_date").shift(-1) - pl.col("settle_date")).dt.total_days(),
        dv_dt_oas=pl.col("flat_rate")
        .truediv(freq)
        .add(1.0)
        .log()
        .mul(freq)
        .mul(pl.col("price")),
        # dv_dt_conv=pl.col("effective_convexity") # EA: (!!!) this is wrong
        dv_dt_conv=pl.col("pde_convexity").mul("price").mul(0.05**2).truediv(2.0).neg(),
        pnl=(pl.col("price").shift(-1) - pl.col("price")).add(
            pl.col("coupon").shift(-1)
        ),
    )
    .with_columns(
        pnl_oas=pl.col("dv_dt_oas").mul(pl.col("ddays")).truediv(360.0),
        pnl_conv=pl.col("dv_dt_conv").mul(pl.col("ddays")).truediv(360.0),
    )
    .with_columns(
        pnl_oas_full=pl.col("pnl_oas") + pl.col("pnl_conv"),
        pnl_err=(pl.col("pnl_oas") + pl.col("pnl_conv")).sub(pl.col("pnl")),
    )
)

res = res.with_columns(
    cpnl=pl.col("pnl").cum_sum(),
    cpnl_oas=pl.col("pnl_oas").cum_sum(),
    cpnl_conv=pl.col("pnl_conv").cum_sum(),
    cpnl_oas_full=pl.col("pnl_oas_full").cum_sum(),
)

In [ ]:
fig = px.line(res[:-4], x="date", y=["cpnl", "cpnl_oas", "cpnl_conv", "cpnl_oas_full"])
fig.update_layout(**layout_dict)

In [ ]:
fig = px.line(res[:-4], x="date", y=["pde_convexity", "effective_convexity"])
fig.update_layout(**layout_dict)

In [ ]:
# 1 cent error for 4 years of data:
res["pnl_err"].abs().sum()

## Rolldown example PnL reconciliation

In [ ]:
bond = get_zc_bond()
sched = ql.MakeSchedule(
    ql.Date(30, 6, 2019),
    ql.Date(15, 9, 2023),  # maturity date will be skipped
    ql.Period("1D"),
    calendar=ql.UnitedStates(ql.UnitedStates.Settlement),
)

res = pl.DataFrame(
    schema=pl.Schema(
        {
            "date": pl.Date,
            "flat_rate": pl.Float64,
            "price": pl.Float64,
            "effective_duration": pl.Float64,
            "effective_convexity": pl.Float64,
        }
    )
)

for i, d in enumerate(sched):
    try:
        row = calc_from_curve(
            bond, d, curve_mode=CurveMode.ROLLDOWN, flat_rate=flat_rate
        )
        res.extend(pl.from_dicts(row))
    except RuntimeError:
        print(d)

res = res.with_columns(
    settle_date=pl.col("date")
    .cast(pl.Int64)
    .add(25569)
    .map_elements(
        lambda x: bond.calendar()
        .advance(ql.Date(x), bond.settlementDays(), ql.Days)
        .to_date(),
        return_dtype=pl.Date,
    )
)

# add coupons
res = res.join(
    pl.from_dicts(
        [
            {"date": cf.date().to_date(), "coupon": cf.amount()}
            for cf in bond.cashflows()
        ]
    ),
    on="date",
    how="left",
)
res = res.with_columns(pl.col("coupon").fill_null(0.0))

In [ ]:
a = 0.2
# this is known from closed form solution for ZCB under Hull White (happens to be identical to rollup example)
res = res.with_columns(
    pde_duration=(
        1.0
        - (
            -a
            * pl.col("settle_date").map_elements(
                lambda x: (zcb.maturityDate().to_date() - x).days
            )
        )
        .truediv(360.0)
        .exp()
    ).truediv(
        a
    ),  # note that duration has opposite sign to the derivative
    pde_convexity=(
        1.0
        - (
            -a
            * pl.col("settle_date").map_elements(
                lambda x: (zcb.maturityDate().to_date() - x).days
            )
        )
        .truediv(360.0)
        .exp()
    )
    .truediv(a)
    .pow(2),
)

In [ ]:
# Check PDE
# V_t + 0.5*s**2*V_xx(x, t) + (theta - a*x) * V_x = x*V
# note that for the first PDE, V_x term is multiplied by 0

sigma = 0.05
freq = 2
res = (
    res.with_columns(
        ddays=(pl.col("settle_date").shift(-1) - pl.col("settle_date")).dt.total_days(),
        dv_dt_oas=pl.col("flat_rate")
        .truediv(freq)
        .add(1.0)
        .log()
        .mul(freq)
        .mul(pl.col("price")),
        dv_dt_dur=pl.col("pde_duration").mul("price").mul(sigma**2 / a),
        # dv_dt_conv=pl.col("effective_convexity") # EA: (!!!) this is wrong
        dv_dt_conv=pl.col("pde_convexity")
        .mul("price")
        .mul(sigma**2)
        .truediv(2.0)
        .neg(),
        pnl=(pl.col("price").shift(-1) - pl.col("price")).add(
            pl.col("coupon").shift(-1)
        ),
    )
    .with_columns(
        pnl_oas=pl.col("dv_dt_oas").mul(pl.col("ddays")).truediv(360.0),
        pnl_dur=pl.col("dv_dt_dur").mul(pl.col("ddays")).truediv(360.0),
        pnl_conv=pl.col("dv_dt_conv").mul(pl.col("ddays")).truediv(360.0),
    )
    .with_columns(
        pnl_oas_full=pl.col("pnl_oas") + pl.col("pnl_dur") + pl.col("pnl_conv"),
        pnl_err=(pl.col("pnl_oas") + pl.col("pnl_dur") + pl.col("pnl_conv")).sub(
            pl.col("pnl")
        ),
    )
)

res = res.with_columns(
    cpnl=pl.col("pnl").cum_sum(),
    cpnl_oas=pl.col("pnl_oas").cum_sum(),
    cpnl_dur=pl.col("pnl_dur").cum_sum(),
    cpnl_conv=pl.col("pnl_conv").cum_sum(),
    cpnl_oas_full=pl.col("pnl_oas_full").cum_sum(),
)

In [ ]:
fig = px.line(
    res[:-4], x="date", y=["cpnl", "cpnl_oas", "cpnl_conv", "cpnl_oas_full", "cpnl_dur"]
)
fig.update_layout(**layout_dict)

In [ ]:
# 6.5 cents error for 4 years of data:
res["pnl_err"].abs().sum()

In [ ]:
fig = px.line(res[:-4], x="date", y=["pnl_err"])
fig.update_layout(**layout_dict)

## Flat at inception PnL reconciliation

In [ ]:
bond = get_zc_bond()
sched = ql.MakeSchedule(
    ql.Date(30, 6, 2019),
    ql.Date(15, 9, 2023),  # maturity date will be skipped
    ql.Period("1D"),
    calendar=ql.UnitedStates(ql.UnitedStates.Settlement),
)

res = pl.DataFrame(
    schema=pl.Schema(
        {
            "date": pl.Date,
            "flat_rate": pl.Float64,
            "price": pl.Float64,
            "effective_duration": pl.Float64,
            "effective_convexity": pl.Float64,
        }
    )
)

for i, d in enumerate(sched):
    try:
        row = calc_from_curve(
            bond, d, curve_mode=CurveMode.FLATINCEPTION, flat_rate=flat_rate
        )
        res.extend(pl.from_dicts(row))
    except RuntimeError:
        print(d)

res = res.with_columns(
    settle_date=pl.col("date")
    .cast(pl.Int64)
    .add(25569)
    .map_elements(
        lambda x: bond.calendar()
        .advance(ql.Date(x), bond.settlementDays(), ql.Days)
        .to_date(),
        return_dtype=pl.Date,
    )
)

# add coupons
res = res.join(
    pl.from_dicts(
        [
            {"date": cf.date().to_date(), "coupon": cf.amount()}
            for cf in bond.cashflows()
        ]
    ),
    on="date",
    how="left",
)
res = res.with_columns(pl.col("coupon").fill_null(0.0))

In [ ]:
a = 0.2
# this is known from closed form solution for ZCB under Hull White (happens to be identical to rollup example)
res = res.with_columns(
    pde_duration=(
        1.0
        - (
            -a
            * pl.col("settle_date").map_elements(
                lambda x: (zcb.maturityDate().to_date() - x).days
            )
        )
        .truediv(360.0)
        .exp()
    ).truediv(
        a
    ),  # note that duration has opposite sign to the derivative
    pde_convexity=(
        1.0
        - (
            -a
            * pl.col("settle_date").map_elements(
                lambda x: (zcb.maturityDate().to_date() - x).days
            )
        )
        .truediv(360.0)
        .exp()
    )
    .truediv(a)
    .pow(2),
)

In [ ]:
# Check PDE
# V_t + 0.5*s**2*V_xx(x, t) + (theta - a*x) * V_x = x*V
# note that for the first PDE, V_x term is multiplied by 0

sigma = 0.05
freq = 2
res = (
    res.with_columns(
        ddays=(pl.col("settle_date").shift(-1) - pl.col("settle_date")).dt.total_days(),
        dv_dt_oas=pl.col("flat_rate")
        .truediv(freq)
        .add(1.0)
        .log()
        .mul(freq)
        .mul(pl.col("price")),
        dv_dt_dur=pl.col("pde_duration")
        .mul("price")
        .mul(sigma**2 / (2.0 * a))
        .mul(
            1.0
            - (
                -2.0
                * a
                * pl.col("settle_date").map_elements(
                    lambda x: (x - zcb.issueDate().to_date()).days
                )
            )
            .truediv(360.0)
            .exp()
        ),
        # dv_dt_conv=pl.col("effective_convexity") # EA: (!!!) this is wrong
        dv_dt_conv=pl.col("pde_convexity")
        .mul("price")  # for zcb price==dirty_price
        .mul(sigma**2)
        .truediv(2.0)
        .neg(),
        pnl=(pl.col("price").shift(-1) - pl.col("price")).add(
            pl.col("coupon").shift(-1)
        ),
    )
    .with_columns(
        pnl_oas=pl.col("dv_dt_oas").mul(pl.col("ddays")).truediv(360.0),
        pnl_dur=pl.col("dv_dt_dur").mul(pl.col("ddays")).truediv(360.0),
        pnl_conv=pl.col("dv_dt_conv").mul(pl.col("ddays")).truediv(360.0),
    )
    .with_columns(
        pnl_oas_full=pl.col("pnl_oas") + pl.col("pnl_dur") + pl.col("pnl_conv"),
        pnl_err=(pl.col("pnl_oas") + pl.col("pnl_dur") + pl.col("pnl_conv")).sub(
            pl.col("pnl")
        ),
    )
)

res = res.with_columns(
    cpnl=pl.col("pnl").cum_sum(),
    cpnl_oas=pl.col("pnl_oas").cum_sum(),
    cpnl_dur=pl.col("pnl_dur").cum_sum(),
    cpnl_conv=pl.col("pnl_conv").cum_sum(),
    cpnl_oas_full=pl.col("pnl_oas_full").cum_sum(),
)

In [ ]:
fig = px.line(
    res[:-4], x="date", y=["cpnl", "cpnl_oas", "cpnl_conv", "cpnl_oas_full", "cpnl_dur"]
)
fig.update_layout(**layout_dict)

In [ ]:
# 1.01 cents error for 4 years of data:
res["pnl_err"].abs().sum()

In [ ]:
fig = px.line(res[:-4], x="date", y=["pnl_err"])
fig.update_layout(**layout_dict)

## Handling PDE changes to preserve flat term structu|re

In [ ]:
def calc_rolling_flat_curve_stats(
    bond: ql.Bond,
    d: ql.Date,
    flat_rate: float | None = None,
    grid_points: int | None = None,
) -> dict:
    ql.Settings.instance().evaluationDate = d
    settle_date = bond.calendar().advance(d, bond.settlementDays(), ql.Days)
    prev_settle_date = bond.calendar().advance(d, bond.settlementDays() - 1, ql.Days)

    if flat_rate is None:
        flat_rate = 0.02
    if grid_points is None:
        grid_points = 100

    dates = ql.MakeSchedule(
        d,
        bond.maturityDate(),  # maturity date will be skipped
        ql.Period("1D"),
        calendar=bond.calendar(),
    )
    dates = [bond.calendar().advance(x, bond.settlementDays(), ql.Days) for x in dates]

    a = 0.2
    sigma = 0.05

    zero_rates = [flat_rate for _ in dates]
    prev_zero_rates = [flat_rate] + [
        flat_rate
        + ((sigma**2) / (4.0 * a**3))
        * (1.0 - np.exp(-2.0 * a * (settle_date - prev_settle_date) / 360.0))
        * np.square(1.0 - np.exp(-a * (d_ - settle_date) / 360.0))
        / ((d_ - settle_date) / 360.0)
        for d_ in dates[1:]
    ]

    curve, prev_curve = (
        ql.ZeroCurve(
            dates,
            z,
            ql.Actual360(),
            bond.calendar(),
            ql.Linear(),
            ql.Compounded,
            ql.Semiannual,
        )
        for z in (zero_rates, prev_zero_rates)
    )

    engine = ql.TreeCallableFixedRateBondEngine(
        ql.HullWhite(ql.YieldTermStructureHandle(prev_curve), a, sigma), grid_points
    )
    bond.setPricingEngine(engine)
    rolled_price = bond.cleanPrice()

    ts_handle = ql.YieldTermStructureHandle(curve)
    engine = ql.TreeCallableFixedRateBondEngine(
        ql.HullWhite(ts_handle, a, sigma), grid_points
    )
    bond.setPricingEngine(engine)

    effective_duration = bond.effectiveDuration(
        0.0, ts_handle, ql.Actual360(), ql.Compounded, ql.Semiannual, 5e-4
    )
    effective_convexity = bond.effectiveConvexity(
        0.0, ts_handle, ql.Actual360(), ql.Compounded, ql.Semiannual, 5e-4
    )

    return {
        "date": d.to_date(),
        "flat_rate": flat_rate,
        "price": bond.cleanPrice(),
        "rolled_price": rolled_price,
        "effective_duration": effective_duration,
        "effective_convexity": effective_convexity,
    }

In [ ]:
flat_rate = 0.01
calc_rolling_flat_curve_stats(zcb, valuation_date, flat_rate=flat_rate)

In [ ]:
bond = get_zc_bond()
sched = ql.MakeSchedule(
    ql.Date(30, 6, 2019),
    ql.Date(15, 9, 2023),  # maturity date will be skipped
    ql.Period("1D"),
    calendar=ql.UnitedStates(ql.UnitedStates.Settlement),
)

res = pl.DataFrame(
    schema=pl.Schema(
        {
            "date": pl.Date,
            "flat_rate": pl.Float64,
            "price": pl.Float64,
            "rolled_price": pl.Float64,
            "effective_duration": pl.Float64,
            "effective_convexity": pl.Float64,
        }
    )
)

for i, d in enumerate(sched):
    try:
        row = calc_rolling_flat_curve_stats(bond, d, flat_rate=flat_rate)
        res.extend(pl.from_dicts(row))
    except RuntimeError:
        print(d)

res = res.with_columns(
    settle_date=pl.col("date")
    .cast(pl.Int64)
    .add(25569)
    .map_elements(
        lambda x: bond.calendar()
        .advance(ql.Date(x), bond.settlementDays(), ql.Days)
        .to_date(),
        return_dtype=pl.Date,
    )
)

# add coupons
res = res.join(
    pl.from_dicts(
        [
            {"date": cf.date().to_date(), "coupon": cf.amount()}
            for cf in bond.cashflows()
        ]
    ),
    on="date",
    how="left",
)
res = res.with_columns(pl.col("coupon").fill_null(0.0))

In [ ]:
a = 0.2
# this is known from closed form solution for ZCB under Hull White (happens to be identical to rollup example)
res = res.with_columns(
    pde_duration=(
        1.0
        - (
            -a
            * pl.col("settle_date").map_elements(
                lambda x: (zcb.maturityDate().to_date() - x).days
            )
        )
        .truediv(360.0)
        .exp()
    ).truediv(
        a
    ),  # note that duration has opposite sign to the derivative
    pde_convexity=(
        1.0
        - (
            -a
            * pl.col("settle_date").map_elements(
                lambda x: (zcb.maturityDate().to_date() - x).days
            )
        )
        .truediv(360.0)
        .exp()
    )
    .truediv(a)
    .pow(2),
)

In [ ]:
# Check PDE
# V_t + 0.5*s**2*V_xx(x, t) + (theta - a*x) * V_x = x*V
# note that for the first PDE, V_x term is multiplied by 0

sigma = 0.05
freq = 2
res = (
    res.with_columns(
        ddays=(pl.col("settle_date").shift(-1) - pl.col("settle_date")).dt.total_days(),
        dv_dt_oas=pl.col("flat_rate")
        .truediv(freq)
        .add(1.0)
        .log()
        .mul(freq)
        .mul(pl.col("price")),
        dv_dt_dur=0.0,
        dv_dt_conv=pl.col("pde_convexity")
        .mul("price")  # for zcb price==dirty_price
        .mul(sigma**2)
        .truediv(2.0)
        .neg(),
        pnl=(pl.col("price").shift(-1) - pl.col("price")).add(
            pl.col("coupon").shift(-1)
        ),
    )
    .with_columns(
        pnl_oas=pl.col("dv_dt_oas").mul(pl.col("ddays")).truediv(360.0),
        pnl_dur=pl.col("dv_dt_dur").mul(pl.col("ddays")).truediv(360.0),
        pnl_conv=pl.col("dv_dt_conv").mul(pl.col("ddays")).truediv(360.0),
        pnl_roll=(pl.col("price") - pl.col("rolled_price")).shift(-1),
    )
    .with_columns(
        pnl_oas_full=pl.col("pnl_oas")
        + pl.col("pnl_dur")
        + pl.col("pnl_conv")
        + pl.col("pnl_roll"),
        pnl_err=(
            pl.col("pnl_oas")
            + pl.col("pnl_dur")
            + pl.col("pnl_conv")
            + pl.col("pnl_roll")
        ).sub(pl.col("pnl")),
    )
)

res = res.with_columns(
    cpnl=pl.col("pnl").cum_sum(),
    cpnl_oas=pl.col("pnl_oas").cum_sum(),
    # cpnl_dur=pl.col("pnl_dur").cum_sum(),
    cpnl_conv=pl.col("pnl_conv").cum_sum(),
    cpnl_roll=pl.col("pnl_roll").cum_sum(),
    cpnl_oas_full=pl.col("pnl_oas_full").cum_sum(),
)

In [ ]:
fig = px.line(
    res[:-4],
    x="date",
    y=["cpnl", "cpnl_oas", "cpnl_conv", "cpnl_oas_full", "cpnl_roll"],
)
fig.update_layout(**layout_dict)

In [ ]:
# 1.62 cents error for 4 years of data:
res["pnl_err"].abs().sum()

In [ ]:
fig = px.line(res[:-4], x="date", y=["pnl_err"])
fig.update_layout(**layout_dict)

## Forward curve reconciliation

In [ ]:
# Rollup example
# valuation_date = ql.Date(15, 9, 2021)
d = valuation_date
dates = ql.MakeSchedule(
    d,
    zcb.maturityDate(),  # maturity date will be skipped
    ql.Period("1D"),
    calendar=zcb.calendar(),
)
dates = list(dates)
a = 0.2
sigma = 0.05
zero_rates = [flat_rate] + [
    flat_rate
    - ((sigma**2) / (2.0 * a**2))
    * (
        ((d_ - d) / 360.0)
        - (2.0 / a) * (1.0 - np.exp(-a * ((d_ - d) / 360.0)))
        + (0.5 / a) * (1.0 - np.exp(-2.0 * a * ((d_ - d) / 360.0)))
    )
    / ((d_ - d) / 360.0)
    for d_ in dates[1:]
]
forwards = [flat_rate] + [
    flat_rate
    - (sigma**2) / (2.0 * a**2) * np.square(1.0 - np.exp(-a * (d_ - d) / 360.0))
    for d_ in dates[1:]
]

curve = ql.ZeroCurve(
    list(dates),
    zero_rates,
    ql.Actual360(),
    zcb.calendar(),
    ql.Linear(),
    ql.Compounded,
    ql.Semiannual,
)
forwards_recon = [
    curve.forwardRate(d_, d_, ql.Actual360(), ql.Compounded, ql.Semiannual).rate()
    for d_ in dates
]

In [ ]:
fig = px.line(
    pl.DataFrame(
        dict(
            date=[d_.to_date() for d_ in dates],
            forward=forwards,
            forward_recon=forwards_recon,
        )
    ),
    "date",
    ["forward", "forward_recon"],
)
fig.update_layout(**layout_dict)
fig.show()

In [ ]:
# Flat at inception
d = zcb.issueDate()
dates = ql.MakeSchedule(
    d,
    zcb.maturityDate(),  # maturity date will be skipped
    ql.Period("1D"),
    calendar=zcb.calendar(),
)
dates = list(dates)
a = 0.2
sigma = 0.05
zero_rates = [flat_rate] + [
    flat_rate
    + ((sigma**2) / (4.0 * a**3))
    * (1.0 - np.exp(-2.0 * a * (d - zcb.issueDate()) / 360.0))
    * np.square(1.0 - np.exp(-a * (d_ - d) / 360.0))
    / ((d_ - d) / 360.0)
    for d_ in dates[1:]
]
forwards = [flat_rate] + [
    flat_rate
    + (sigma**2)
    / (2 * a**2)
    * (1.0 - np.exp(-a * (d_ - d) / 360.0))
    * np.exp(-a * (d_ - d) / 360.0)
    * (1.0 - np.exp(-2.0 * a * (d - zcb.issueDate()) / 360.0))
    for d_ in dates[1:]
]

curve = ql.ZeroCurve(
    list(dates),
    zero_rates,
    ql.Actual360(),
    zcb.calendar(),
    ql.Linear(),
    ql.Compounded,
    ql.Semiannual,
)
forwards_recon = [
    curve.forwardRate(d_, d_, ql.Actual360(), ql.Compounded, ql.Semiannual).rate()
    for d_ in dates
]

In [ ]:
fig = px.line(
    pl.DataFrame(
        dict(
            date=[d_.to_date() for d_ in dates],
            forward=forwards,
            forward_recon=forwards_recon,
        )
    ),
    "date",
    ["forward", "forward_recon"],
)
fig.update_layout(**layout_dict)
fig.show()

# Fixed rate bond carry with yield & z-spread

In [ ]:
def get_fr_bond() -> ql.FixedRateBond:
    settlement_days = 2
    face_amount = 100

    issue_date = ql.Date(30, 6, 2019)
    maturity_date = ql.Date(15, 9, 2023)
    tenor = ql.Period(ql.Semiannual)
    calendar = ql.UnitedStates(ql.UnitedStates.Settlement)
    accrual_convention = ql.Unadjusted
    Rule = ql.DateGeneration.Backward
    endofMonth = False
    schedule = ql.Schedule(
        issue_date,
        maturity_date,
        tenor,
        calendar,
        accrual_convention,
        accrual_convention,
        Rule,
        endofMonth,
    )

    coupon = 0.07
    day_counter = ql.Actual360()  # ql.ActualActual(ql.ActualActual.Bond)
    payment_convention = ql.Following

    bond = ql.FixedRateBond(
        settlement_days,
        face_amount,
        schedule,
        [coupon],
        day_counter,
        payment_convention,
    )
    return bond

In [ ]:
def calc_from_yield(bond, d, y):
    # y1 = ql.BondFunctions.bondYield(bond, ql.BondPrice(101., ql.BondPrice.Clean), ql.Actual360(), ql.Compounded, ql.Annual, ql.Date(30, 6, 2020)) # px -> yield
    # y2 = ql.BondFunctions.bondYield(bond, ql.BondPrice(101., ql.BondPrice.Dirty), ql.Actual360(), ql.Compounded, ql.Annual, ql.Date(30, 6, 2020)) # dpx -> yield
    px = bond.cleanPrice(
        y,
        ql.Actual360(),
        ql.Compounded,
        ql.Semiannual,
        d,
    )  # yield -> px
    dpx = bond.dirtyPrice(
        y,
        ql.Actual360(),
        ql.Compounded,
        ql.Semiannual,
        d,
    )  # yield -> dpx
    dur = ql.BondFunctions.duration(
        bond,
        y,
        ql.Actual360(),
        ql.Compounded,
        ql.Semiannual,
        ql.Duration.Modified,
        d,
    )
    return {
        "date": d.to_date(),
        "yield": y,
        "price": px,
        "dirty_price": dpx,
        "duration": dur,
    }

In [ ]:
def calc_from_zspread(bond, d, zs, flat_rate=None) -> dict:
    ql.Settings.instance().evaluationDate = d

    if flat_rate is None:
        flat_rate = 0.02
    curve = ql.FlatForward(
        bond.settlementDays(),
        ql.UnitedStates(ql.UnitedStates.Settlement),
        ql.QuoteHandle(ql.SimpleQuote(flat_rate)),
        ql.Actual360(),
        ql.Compounded,
        ql.Semiannual,
    )

    # zs = ql.BondFunctions.zSpread(
    #     bond,
    #     ql.BondPrice(101, ql.BondPrice.Clean),
    #     curve,
    #     ql.Actual360(),
    #     ql.Compounded,
    #     ql.Semiannual,
    #     bond.calendar().advance(d, bond.settlementDays(), ql.Days),
    # ) # px -> zs

    dzss = [-0.0005, 0.0, 0.0005]
    pxs = []
    dpxs = []
    for dzs in dzss:
        zspread = ql.SimpleQuote(zs + dzs)
        spread_handle = ql.QuoteHandle(zspread)
        ts_handle = ql.YieldTermStructureHandle(curve)
        ts_spreaded = ql.ZeroSpreadedTermStructure(
            ts_handle,
            spread_handle,
            ql.Compounded,
            ql.Semiannual,
            ql.Actual360(),
        )
        ts_spreaded_handle = ql.YieldTermStructureHandle(ts_spreaded)
        bond_engine = ql.DiscountingBondEngine(ts_spreaded_handle)
        bond.setPricingEngine(bond_engine)
        pxs.append(bond.cleanPrice())
        dpxs.append(bond.dirtyPrice())

    spread_duration = (
        ((dpxs[0] - dpxs[-1]) / (dzss[-1] - dzss[0])) / dpxs[1]
        if dpxs[1] > 0.0
        else None
    )
    spread_convexity = (
        (dpxs[0] - 2 * dpxs[1] + dpxs[2]) / (dpxs[1] * (dzss[1] - dzss[0]) ** 2)
        if dpxs[1] > 0.0
        else None
    )

    return {
        "date": d.to_date(),
        "zspread": zs,
        "flat_rate": flat_rate,
        "price": pxs[1],
        "dirty_price": dpxs[1],
        "spread_duration": spread_duration,
        "spread_convexity": spread_convexity,
    }

In [ ]:
def calc_from_price(bond, d, price, flat_rate=None) -> dict:
    ql.Settings.instance().evaluationDate = d

    if flat_rate is None:
        flat_rate = 0.02
    curve = ql.FlatForward(
        bond.settlementDays(),
        ql.UnitedStates(ql.UnitedStates.Settlement),
        ql.QuoteHandle(ql.SimpleQuote(flat_rate)),
        ql.Actual360(),
        ql.Compounded,
        ql.Semiannual,
    )

    zs = ql.BondFunctions.zSpread(
        bond,
        ql.BondPrice(price, ql.BondPrice.Clean),
        curve,
        ql.Actual360(),
        ql.Compounded,
        ql.Semiannual,
        bond.calendar().advance(d, bond.settlementDays(), ql.Days),
    )  # px -> zs

    dzss = [-0.0005, 0.0, 0.0005]
    pxs = []
    dpxs = []
    for dzs in dzss:
        zspread = ql.SimpleQuote(zs + dzs)
        spread_handle = ql.QuoteHandle(zspread)
        ts_handle = ql.YieldTermStructureHandle(curve)
        ts_spreaded = ql.ZeroSpreadedTermStructure(
            ts_handle,
            spread_handle,
            ql.Compounded,
            ql.Semiannual,
            ql.Actual360(),
        )
        ts_spreaded_handle = ql.YieldTermStructureHandle(ts_spreaded)
        bond_engine = ql.DiscountingBondEngine(ts_spreaded_handle)
        bond.setPricingEngine(bond_engine)
        pxs.append(bond.cleanPrice())
        dpxs.append(bond.dirtyPrice())

    spread_duration = (
        ((dpxs[0] - dpxs[-1]) / (dzss[-1] - dzss[0])) / dpxs[1]
        if dpxs[1] > 0.0
        else None
    )
    spread_convexity = (
        (dpxs[0] - 2 * dpxs[1] + dpxs[2]) / (dpxs[1] * (dzss[1] - dzss[0]) ** 2)
        if dpxs[1] > 0.0
        else None
    )

    return {
        "date": d.to_date(),
        "zspread": zs,
        "flat_rate": flat_rate,
        "price": price,
        "dirty_price": dpxs[1],
        "spread_duration": spread_duration,
        "spread_convexity": spread_convexity,
    }

In [ ]:
# yield check
bond = get_fr_bond()
yld = 0.06
sched = ql.MakeSchedule(
    ql.Date(30, 6, 2019),
    ql.Date(15, 9, 2023),  # maturity date will be skipped
    ql.Period("1D"),
    calendar=ql.UnitedStates(ql.UnitedStates.Settlement),
)

res = pl.DataFrame(
    schema=pl.Schema(
        {
            "date": pl.Date,
            "yield": pl.Float64,
            "price": pl.Float64,
            "dirty_price": pl.Float64,
            "duration": pl.Float64,
        }
    )
)

for i, d in enumerate(sched):
    try:
        row = calc_from_yield(bond, d, yld)
        res.extend(pl.from_dicts(row))
    except RuntimeError:
        print(d)

# add coupons
res = res.join(
    pl.from_dicts(
        [
            {"date": cf.date().to_date(), "coupon": cf.amount()}
            for cf in bond.cashflows()
        ]
    ),
    on="date",
    how="left",
)
res = res.with_columns(pl.col("coupon").fill_null(0.0))

In [ ]:
freq = 2
res = res.with_columns(
    ddays=(pl.col("date").shift(-1) - pl.col("date")).dt.total_days(),
    dv_dt=pl.col("yield")
    .truediv(freq)
    .add(1.0)
    .log()
    .mul(freq)
    .mul(pl.col("dirty_price")),
    pnl=(pl.col("dirty_price").shift(-1) - pl.col("dirty_price")).add(
        pl.col("coupon").shift(-1)
    ),
).with_columns(err=pl.col("dv_dt").mul(pl.col("ddays")).truediv(360.0).sub("pnl"))

In [ ]:
res["err"].sum()  # 0.43 cents on 100 for 4 years.. very close <-> 0.000043
res["err"].abs().sum()  # actually always the same sign

In [ ]:
# zspread check
bond = get_fr_bond()
zs = 0.04
sched = ql.MakeSchedule(
    ql.Date(30, 6, 2019),
    ql.Date(15, 9, 2023),  # maturity date will be skipped
    ql.Period("1D"),
    calendar=ql.UnitedStates(ql.UnitedStates.Settlement),
)

res = pl.DataFrame(
    schema=pl.Schema(
        {
            "date": pl.Date,
            "zspread": pl.Float64,
            "flat_rate": pl.Float64,
            "price": pl.Float64,
            "dirty_price": pl.Float64,
            "spread_duration": pl.Float64,
            "spread_convexity": pl.Float64,
        }
    )
)

for i, d in enumerate(sched):
    try:
        row = calc_from_zspread(bond, d, zs)
        res.extend(pl.from_dicts(row))
    except RuntimeError:
        print(d)

res = res.with_columns(
    settle_date=pl.col("date")
    .cast(pl.Int64)
    .add(25569)
    .map_elements(
        lambda x: bond.calendar()
        .advance(ql.Date(x), bond.settlementDays(), ql.Days)
        .to_date(),
        return_dtype=pl.Date,
    )
)

# add coupons
res = res.join(
    pl.from_dicts(
        [
            {"settle_date": cf.date().to_date(), "coupon": cf.amount()}
            for cf in bond.cashflows()
        ]
    )
    .group_by("settle_date")
    .agg(pl.col("coupon").sum()),
    on="settle_date",
    how="left",
)
res = res.with_columns(pl.col("coupon").fill_null(0.0))

In [ ]:
freq = 2
res = res.with_columns(
    ddays=(pl.col("settle_date").shift(-1) - pl.col("settle_date")).dt.total_days(),
    dv_dt=pl.col("zspread")
    .add(pl.col("flat_rate"))
    .truediv(freq)
    .add(1.0)
    .log()
    .mul(freq)
    .mul(pl.col("dirty_price")),
    pnl=(pl.col("dirty_price").shift(-1) - pl.col("dirty_price")).add(
        pl.col("coupon").shift(-1)
    ),
).with_columns(err=pl.col("dv_dt").mul(pl.col("ddays")).truediv(360.0).sub("pnl"))

In [ ]:
res["err"].sum()  # 0.43 cents on 100 for 4 years.. very close <-> 0.000043
res["err"].abs().sum()  # actually always the same sign

In [ ]:
fig = px.line(res[:-4], x="date", y=["price", "dirty_price"])
fig.update_layout(**layout_dict)

In [ ]:
# price check
bond = get_fr_bond()
# bond = get_sample_bond()
price = 100.2
sched = ql.MakeSchedule(
    ql.Date(30, 6, 2019),
    ql.Date(15, 9, 2023),  # maturity date will be skipped
    ql.Period("1D"),
    calendar=ql.UnitedStates(ql.UnitedStates.Settlement),
)

res = pl.DataFrame(
    schema=pl.Schema(
        {
            "date": pl.Date,
            "zspread": pl.Float64,
            "flat_rate": pl.Float64,
            "price": pl.Float64,
            "dirty_price": pl.Float64,
            "spread_duration": pl.Float64,
            "spread_convexity": pl.Float64,
        }
    )
)

for i, d in enumerate(sched):
    try:
        row = calc_from_price(bond, d, price)
        res.extend(pl.from_dicts(row))
    except RuntimeError:
        print(d)

res = res.with_columns(
    settle_date=pl.col("date")
    .cast(pl.Int64)
    .add(25569)
    .map_elements(
        lambda x: bond.calendar()
        .advance(ql.Date(x), bond.settlementDays(), ql.Days)
        .to_date(),
        return_dtype=pl.Date,
    )
)

# add coupons
res = res.join(
    pl.from_dicts(
        [
            {"settle_date": cf.date().to_date(), "coupon": cf.amount()}
            for cf in bond.cashflows()
        ]
    )
    .group_by("settle_date")
    .agg(pl.col("coupon").sum()),
    on="settle_date",
    how="left",
)
res = res.with_columns(pl.col("coupon").fill_null(0.0))

In [ ]:
freq = 2
res = res.with_columns(
    ddays=(pl.col("settle_date").shift(-1) - pl.col("settle_date")).dt.total_days(),
    dv_dt=pl.col("zspread")
    .add(pl.col("flat_rate"))
    .truediv(freq)
    .add(1.0)
    .log()
    .mul(freq)
    .mul(pl.col("dirty_price")),
    dz=(pl.col("zspread").shift(-1) - pl.col("zspread")),
    pnl=(pl.col("dirty_price").shift(-1) - pl.col("dirty_price")).add(
        pl.col("coupon").shift(-1)
    ),
).with_columns(
    err=pl.col("dv_dt")
    .mul(pl.col("ddays"))
    .truediv(360.0)
    .sub(pl.col("spread_duration").mul(pl.col("dirty_price")).mul(pl.col("dz")))
    .sub(
        pl.col("spread_convexity")
        .mul(pl.col("dirty_price"))
        .mul(pl.col("dz").pow(2))
        .truediv(2.0)
    )
    .sub("pnl")
)

In [ ]:
res["err"][
    :-20
].sum()  # 1.46 cent on 100 for 3 years.. very close <-> 0.0001 precision (1bp)
res["err"][:-20].abs().sum()  # actually always the same sign

In [ ]:
fig = px.line(res[:-20], x="date", y=["zspread"])
fig.update_layout(**layout_dict)

# Callable bond carry with OAS

In [ ]:
def get_callable_bond() -> ql.CallableFixedRateBond:
    settlement_days = 2
    face_amount = 100

    issue_date = ql.Date(30, 6, 2019)
    maturity_date = ql.Date(15, 9, 2023)
    tenor = ql.Period(ql.Semiannual)
    calendar = ql.UnitedStates(ql.UnitedStates.Settlement)
    accrual_convention = ql.Unadjusted
    Rule = ql.DateGeneration.Backward
    endofMonth = False
    schedule = ql.Schedule(
        issue_date,
        maturity_date,
        tenor,
        calendar,
        accrual_convention,
        accrual_convention,
        Rule,
        endofMonth,
    )

    coupon = 0.07
    day_count = ql.Actual360()  # ql.ActualActual(ql.ActualActual.Bond)

    callability_schedule = ql.CallabilitySchedule()
    for strike, dt in zip(
        [106.0, 104.0, 102.0, 101.0, 100.0],
        [
            ql.Date(15, 9, 2019),
            ql.Date(15, 9, 2020),
            ql.Date(15, 9, 2021),
            ql.Date(15, 9, 2022),
            ql.Date(15, 9, 2023),
        ],
    ):
        callability_schedule.append(
            ql.Callability(
                ql.BondPrice(strike, ql.BondPrice.Clean), ql.Callability.Call, dt
            )
        )

    bond = ql.CallableFixedRateBond(
        settlement_days,
        face_amount,
        schedule,
        [coupon],
        day_count,
        ql.Following,
        face_amount,  # redemption
        issue_date,
        callability_schedule,
    )

    return bond

In [ ]:
def calc_from_oas(
    bond: ql.CallableFixedRateBond,
    d: ql.Date,
    oas: float | None,
    flat_rate: float | None = None,
    grid_points: int | None = None,
    return_zspread_sensi: bool = False,
) -> dict:
    ql.Settings.instance().evaluationDate = d

    if flat_rate is None:
        flat_rate = 0.02

    curve = ql.FlatForward(
        bond.settlementDays(),
        ql.UnitedStates(ql.UnitedStates.Settlement),
        ql.QuoteHandle(ql.SimpleQuote(flat_rate)),
        ql.Actual360(),
    )
    ts_handle = ql.YieldTermStructureHandle(curve)

    a = 0.2
    sigma = 0.05
    if grid_points is None:
        grid_points = 100
    engine = ql.TreeCallableFixedRateBondEngine(
        ql.HullWhite(ts_handle, a, sigma), grid_points
    )
    bond.setPricingEngine(engine)

    doass = [-0.0005, 0.0, 0.0005]
    pxs = [None] * 3
    dpxs = [None] * 3
    for i, doas in enumerate(doass):
        if doas != 0.0:
            continue
        price = bond.cleanPriceOAS(
            oas + doas,
            ts_handle,
            ql.Actual360(),
            ql.Compounded,
            ql.Semiannual,
            bond.calendar().advance(d, bond.settlementDays(), ql.Days),
        )
        accrued = bond.accruedAmount(
            bond.calendar().advance(d, bond.settlementDays(), ql.Days)
        )
        pxs[i] = price
        dpxs[i] = price + accrued

    effective_duration = bond.effectiveDuration(
        oas, ts_handle, ql.Actual360(), ql.Compounded, ql.Semiannual, 5e-4
    )
    effective_convexity = bond.effectiveConvexity(
        oas, ts_handle, ql.Actual360(), ql.Compounded, ql.Semiannual, 5e-4
    )
    # empirical_duration = (
    #     ((dpxs[0] - dpxs[-1]) / (doass[-1] - doass[0])) / dpxs[1]
    #     if dpxs[1] > 0.0
    #     else None
    # )
    # empirical_convexity = (
    #         (dpxs[0] - 2 * dpxs[1] + dpxs[2]) / (dpxs[1] * (doass[1] - doass[0]) ** 2)
    #         if dpxs[1] > 0.0
    #         else None
    # )

    zs = ql.BondFunctions.zSpread(
        bond,
        ql.BondPrice(pxs[1], ql.BondPrice.Clean),
        curve,
        ql.Actual360(),
        ql.Compounded,
        ql.Semiannual,
        bond.calendar().advance(d, bond.settlementDays(), ql.Days),
    )  # px -> zs

    if return_zspread_sensi:
        # set sigma to small number --> zspread
        engine = ql.TreeCallableFixedRateBondEngine(
            ql.HullWhite(ts_handle, a, 1e-5), grid_points
        )
        bond.setPricingEngine(engine)
        spread_duration = bond.effectiveDuration(
            zs, ts_handle, ql.Actual360(), ql.Compounded, ql.Semiannual, 5e-4
        )
        spread_convexity = bond.effectiveConvexity(
            zs, ts_handle, ql.Actual360(), ql.Compounded, ql.Semiannual, 5e-4
        )
    else:
        spread_duration = None
        spread_convexity = None

    # dzss = [-0.0005, 0.0, 0.0005]
    # dpxs1 = []
    # for dzs in dzss:
    #     price = bond.cleanPriceOAS(
    #         zs + dzs, ts_handle, ql.Actual360(), ql.Compounded, ql.Semiannual
    #     )
    #     dpxs1.append(price + accrued)

    # spread_duration1 = (
    #     ((dpxs1[0] - dpxs1[-1]) / (dzss[-1] - dzss[0])) / dpxs1[1]
    #     if dpxs1[1] > 0.0
    #     else None
    # )
    # spread_convexity1 = (
    #     (dpxs1[0] - 2 * dpxs1[1] + dpxs1[2]) / (dpxs1[1] * (dzss[1] - dzss[0]) ** 2)
    #     if dpxs1[1] > 0.0
    #     else None
    # )

    # bond_yield = bond.bondYield(
    #     ql.BondPrice(100, ql.BondPrice.Clean), day_count, compounding, frequency
    # )
    # if oas is None:
    #     oas = bond.OAS(100.0, ts_handle, ql.Actual360(), ql.Compounded, ql.Semiannual, )

    return {
        "date": d.to_date(),
        "oas": oas,
        "zspread": zs,
        "flat_rate": flat_rate,
        "price": pxs[1],
        "dirty_price": dpxs[1],
        "effective_duration": effective_duration,
        # "empirical_duration": empirical_duration,
        "effective_convexity": effective_convexity,
        # "empirical_convexity": empirical_convexity,
        "spread_duration": spread_duration,
        # "spread_duration1": spread_duration1,
        "spread_convexity": spread_convexity,
        # "spread_convexity1":spread_convexity1,
        "accrued": accrued,
    }

In [ ]:
bond = get_callable_bond()
calc_from_oas(bond, ql.Date(30, 6, 2019), 0.0070)

In [ ]:
# zspread
bond = get_callable_bond()
oas = np.linspace(-0.03 + 1e-4, 0.21, 600)
prices = np.empty(600, float)
for i, o in enumerate(oas):
    prices[i] = calc_from_oas(bond, ql.Date(30, 6, 2019), o)["price"]
fig = px.line(pl.DataFrame(dict(oas=oas, price=prices)), "oas", "price")
fig.update_layout(**layout_dict)

In [ ]:
# oas check
bond = get_callable_bond()
oas = 0.0425
sched = ql.MakeSchedule(
    ql.Date(30, 6, 2019),
    ql.Date(15, 9, 2023),  # maturity date will be skipped
    ql.Period("1D"),
    calendar=ql.UnitedStates(ql.UnitedStates.Settlement),
)

res = pl.DataFrame(
    schema=pl.Schema(
        {
            "date": pl.Date,
            "oas": pl.Float64,
            "zspread": pl.Float64,
            "flat_rate": pl.Float64,
            "price": pl.Float64,
            "dirty_price": pl.Float64,
            "effective_duration": pl.Float64,
            # "empirical_duration": pl.Float64,
            "effective_convexity": pl.Float64,
            # "empirical_convexity": pl.Float64,
            "spread_duration": pl.Float64,
            "spread_convexity": pl.Float64,
            "accrued": pl.Float64,
        }
    )
)

for i, d in enumerate(sched):
    try:
        row = calc_from_oas(bond, d, oas, return_zspread_sensi=True)
        res.extend(pl.from_dicts(row))
    except RuntimeError:
        print(d)

res = res.with_columns(
    settle_date=pl.col("date")
    .cast(pl.Int64)
    .add(25569)
    .map_elements(
        lambda x: bond.calendar()
        .advance(ql.Date(x), bond.settlementDays(), ql.Days)
        .to_date(),
        return_dtype=pl.Date,
    )
)

# add coupons
res = res.join(
    pl.from_dicts(
        [
            {"settle_date": cf.date().to_date(), "coupon": cf.amount()}
            for cf in bond.cashflows()
        ]
    )
    .group_by("settle_date")
    .agg(pl.col("coupon").sum()),
    on="settle_date",
    how="left",
)
res = res.with_columns(pl.col("coupon").fill_null(0.0))

In [ ]:
freq = 2
res = res.with_columns(
    ddays=(pl.col("settle_date").shift(-1) - pl.col("settle_date")).dt.total_days(),
    dv_dt=pl.col("zspread")
    .add(pl.col("flat_rate"))
    .truediv(freq)
    .add(1.0)
    .log()
    .mul(freq)
    .mul(pl.col("dirty_price")),
    dz=(pl.col("zspread").shift(-1) - pl.col("zspread")),
    pnl=(pl.col("dirty_price").shift(-1) - pl.col("dirty_price")).add(
        pl.col("coupon").shift(-1)
    ),
).with_columns(
    err=pl.col("dv_dt")
    .mul(pl.col("ddays"))
    .truediv(360.0)
    .sub(pl.col("spread_duration").mul(pl.col("dirty_price")).mul(pl.col("dz")))
    .sub(
        pl.col("spread_convexity")
        .mul(pl.col("dirty_price"))
        .mul(pl.col("dz").pow(2))
        .truediv(2.0)
    )
    .sub("pnl")
)

res.write_parquet("/data/constant_oas_run.parquet")

In [ ]:
res = pl.read_parquet("/data/constant_oas_run.parquet")

In [ ]:
res["err"].sum()  # 6 cents on 100 for 4 years.. very close <-> 0.000043
res["err"].abs().sum()  # actually always the same sign

In [ ]:
fig = px.line(res[:-4], x="date", y=["oas", "zspread"])
fig.update_layout(**layout_dict)

In [ ]:
fig = px.line(res[:-4], x="date", y=["price", "dirty_price"])
fig.update_layout(**layout_dict)

In [ ]:
def calc_sensi_oas(
    bond: ql.CallableFixedRateBond,
    d: ql.Date,
    oas: float | None,
    flat_rate: float | None = None,
    grid_points: int | None = None,
) -> dict:
    ql.Settings.instance().evaluationDate = d

    if flat_rate is None:
        flat_rate = 0.02

    curve = ql.FlatForward(
        bond.settlementDays(),
        ql.UnitedStates(ql.UnitedStates.Settlement),
        ql.QuoteHandle(ql.SimpleQuote(flat_rate)),
        ql.Actual360(),
    )
    ts_handle = ql.YieldTermStructureHandle(curve)

    a = 0.2
    sigma = 0.05
    if grid_points is None:
        grid_points = 100
    engine = ql.TreeCallableFixedRateBondEngine(
        ql.HullWhite(ts_handle, a, sigma), grid_points
    )
    bond.setPricingEngine(engine)

    effective_duration = bond.effectiveDuration(
        oas, ts_handle, ql.Actual360(), ql.Compounded, ql.Semiannual, 5e-4
    )
    effective_convexity = bond.effectiveConvexity(
        oas, ts_handle, ql.Actual360(), ql.Compounded, ql.Semiannual, 5e-4
    )

    return {
        "date": d.to_date(),
        "oas": oas,
        "flat_rate": flat_rate,
        # 'price': price,
        # 'dirty_price': price+accrued,
        "effective_duration_exact": effective_duration,
        "effective_convexity_exact": effective_convexity,
    }

In [ ]:
calc_sensi_oas(bond, ql.Date(30, 6, 2020), 0.0425, grid_points=1000)

In [ ]:
# oas check
bond = get_callable_bond()
oas = 0.0425
sched = ql.MakeSchedule(
    ql.Date(30, 6, 2019),
    ql.Date(15, 9, 2023),  # maturity date will be skipped
    ql.Period("1D"),
    calendar=ql.UnitedStates(ql.UnitedStates.Settlement),
)

sensi = pl.DataFrame(
    schema=pl.Schema(
        {
            "date": pl.Date,
            "oas": pl.Float64,
            "flat_rate": pl.Float64,
            "effective_duration_exact": pl.Float64,
            "effective_convexity_exact": pl.Float64,
        }
    )
)

for i, d in enumerate(sched):
    if 0 != i % 4:
        continue
    try:
        row = calc_sensi_oas(bond, d, oas, grid_points=1000)
        sensi.extend(pl.from_dicts(row))
    except RuntimeError:
        print(d)

sensi.write_parquet("/data/constant_oas_sensi.parquet")

In [ ]:
sensi = pl.read_parquet("/data/constant_oas_sensi.parquet")

In [ ]:
# amend res with sensi. Interpolate linearly between calculated values (it takes a long time to calculate exact values)
res = res.join(
    sensi.select(
        pl.col("date", "effective_duration_exact", "effective_convexity_exact")
    ),
    on="date",
    how="left",
).with_columns(
    pl.col("effective_duration_exact", "effective_convexity_exact").interpolate()
)  # this

In [ ]:
fig = px.line(
    res[:-4], x="date", y=["effective_convexity", "effective_convexity_exact"]
)
fig.update_layout(**layout_dict)

In [ ]:
freq = 2
res = (
    res.with_columns(
        dv_dt_oas=pl.col("oas")
        .add(pl.col("flat_rate"))
        .truediv(freq)
        .add(1.0)
        .log()
        .mul(freq)
        .mul(pl.col("dirty_price")),
        dv_dt_conv=pl.col("effective_convexity_exact")
        .mul("dirty_price")
        .mul(0.05**2)
        .truediv(2.0)
        .neg(),
    )
    .with_columns(
        pnl_oas=pl.col("dv_dt_oas").mul(pl.col("ddays")).truediv(360.0),
        pnl_conv=pl.col("dv_dt_conv").mul(pl.col("ddays")).truediv(360.0),
    )
    .with_columns(
        pnl_oas_full=pl.col("pnl_oas") + pl.col("pnl_conv"),
        err_oas=(pl.col("pnl_oas") + pl.col("pnl_conv")).sub(pl.col("pnl")),
    )
)

res = res.with_columns(
    cpnl=pl.col("pnl").cum_sum(),
    cpnl_oas=pl.col("pnl_oas").cum_sum(),
    cpnl_conv=pl.col("pnl_conv").cum_sum(),
    cpnl_oas_full=pl.col("pnl_oas_full").cum_sum(),
)